In [1]:
!pip install -q -U transformers datasets accelerate peft trl bitsandbytes
!pip install -q -U huggingface_hub sentencepiece scikit-learn

In [2]:
import torch
import time
import gc
import re
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print("All imports loaded!")

ModuleNotFoundError: No module named 'trl'

In [3]:
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv("Assassin.csv", low_memory=False)
df = df.dropna(subset=['body', 'subject'])  # Remove rows with missing body/subject
df['body'] = df['body'].astype(str)
df['subject'] = df['subject'].astype(str)

# Truncate very long emails to keep within context limits
# Average email ~1500 chars, we cap at 1500 to be safe with tokens
df['body_truncated'] = df['body'].str[:1500]

print(f"Total emails: {len(df)}")
print(f"Phishing: {(df['label']==1).sum()} | Benign: {(df['label']==0).sum()}")

# --- STRATIFIED SAMPLING ---
# We want 400 for training, 100 for testing
# Stratified = keeps the same phishing:benign ratio

# First get a manageable subset
subset, _ = train_test_split(
    df, train_size=500, stratify=df['label'], random_state=42
)

# Now split into train/test
train_df, test_df = train_test_split(
    subset, test_size=0.2, stratify=subset['label'], random_state=42
)

print(f"\nTrain set: {len(train_df)} (Phishing: {(train_df['label']==1).sum()}, Benign: {(train_df['label']==0).sum()})")
print(f"Test set:  {len(test_df)} (Phishing: {(test_df['label']==1).sum()}, Benign: {(test_df['label']==0).sum()})")

Total emails: 5792
Phishing: 1705 | Benign: 4087

Train set: 400 (Phishing: 118, Benign: 282)
Test set:  100 (Phishing: 29, Benign: 71)


In [4]:
SYSTEM_PROMPT = """You are an expert email security analyst. When given an email, classify it as either PHISHING or BENIGN.

Rules:
- Respond with EXACTLY one line: "Classification: PHISHING" or "Classification: BENIGN"
- Then provide a brief explanation (2-3 sentences max)
- Base your analysis on: sender legitimacy, subject line, urgency language, suspicious links, requests for personal info, and overall tone"""

def email_to_prompt(row):
    """Convert a raw email into a structured prompt."""
    sender = str(row.get('sender', 'Unknown'))
    subject = str(row.get('subject', '(no subject)'))
    body = str(row.get('body_truncated', ''))

    prompt = f"""Analyze this email:

From: {sender}
Subject: {subject}

Body:
{body}"""
    return prompt


def email_to_response(row):
    """Generate the expected response for a training example."""
    label = row['label']
    subject = str(row.get('subject', ''))
    body = str(row.get('body_truncated', ''))[:300]

    if label == 1:  # Phishing
        # Generate explanation based on common phishing signals
        reasons = []
        urgency_words = ['urgent', 'immediately', 'act now', 'limited time',
                         'free', 'winner', 'congratulations', 'verify', 'click here']
        found_words = [w for w in urgency_words if w.lower() in (subject + ' ' + body).lower()]
        if found_words:
            reasons.append(f"Uses urgency/manipulation language: {', '.join(found_words[:3])}")
        if any(x in body.lower() for x in ['http://', 'click here', 'click below', 'click this']):
            reasons.append("Contains suspicious links prompting the user to click")
        if any(x in body.lower() for x in ['credit card', 'ssn', 'password', 'bank account', 'social security']):
            reasons.append("Requests sensitive personal or financial information")
        if any(x in body.lower() for x in ['$', 'money', 'earn', 'cash', 'profit', 'income']):
            reasons.append("Contains financial lures or get-rich-quick language")
        if not reasons:
            reasons.append("Email exhibits characteristics typical of spam/phishing campaigns")
            reasons.append("Unsolicited commercial content with deceptive intent")

        explanation = ". ".join(reasons[:3]) + "."
        return f"Classification: PHISHING\n\nExplanation: {explanation}"
    else:  # Benign
        return f"Classification: BENIGN\n\nExplanation: This appears to be a legitimate email. The content is consistent with normal communication — no urgency manipulation, no suspicious links requesting credentials, and no hallmarks of a phishing or spam campaign."


# Test the formatting
sample = train_df.iloc[0]
print("=== SAMPLE FORMATTED INPUT ===")
print(email_to_prompt(sample)[:500])
print("\n=== SAMPLE FORMATTED OUTPUT ===")
print(email_to_response(sample))

=== SAMPLE FORMATTED INPUT ===
Analyze this email:

From: ankara@dunyagazetesi.com.tr
Subject: Kime Oy Vereceksiniz ?

Body:
=DDyi g=FCnler

=20
D=DCNYA Gazetesi, i=E7inde bulundu=F0umuz siyasi karma=FEa d=F6neminin se=
=E7imler sonras=FDnda nas=FDl bir hal alaca=F0=FD konusunda kapsaml=FD bi=
r ara=FEt=FDrma yapmaktad=FDr. Bu =E7er=E7evede toplumumuzun m=FCmk=FCn o=
ldu=F0unca geni=FE bir kesiminin g=F6r=FC=FElerine ba=FEvurmay=FD gerekli=
 g=F6rd=FCk.=20

=20
3 Kas=FDm 2002 tarihinde yap=FDlmas=FD =F6ng=F6r=FClen se=E7imler

=== SAMPLE FORMATTED OUTPUT ===
Classification: PHISHING

Explanation: Email exhibits characteristics typical of spam/phishing campaigns. Unsolicited commercial content with deceptive intent.


In [5]:
# Store all results for final comparison
ALL_RESULTS = {}

def load_model(model_name):
    """Load a model with 4-bit quantization."""
    print(f"\n{'='*60}")
    print(f"Loading: {model_name}")
    print(f"{'='*60}")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        model.config.pad_token_id = model.config.eos_token_id

    params = sum(p.numel() for p in model.parameters())
    print(f"Loaded! Parameters: {params:,}")
    return model, tokenizer


def ask(model, tokenizer, user_input, max_tokens=200):
    """Generate a response from the model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT + "\n/no_think"},
        {"role": "user", "content": user_input},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, max_length=2048
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.2,     # Prevents repetitive loops
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    # Strip thinking tags if present
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    # Also handle unclosed think tags (model got stuck)
    response = re.sub(r'<think>.*', '', response, flags=re.DOTALL).strip()
    return response


def extract_prediction(response):
    """Extract PHISHING or BENIGN from model response."""
    # Clean up the response first
    response_clean = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL)
    response_clean = re.sub(r'<think>.*', '', response_clean, flags=re.DOTALL)
    response_upper = response_clean.upper().strip()

    if not response_upper:
        return -1

    # Look for explicit classification
    if "CLASSIFICATION: PHISHING" in response_upper or "CLASSIFICATION:PHISHING" in response_upper:
        return 1
    if "CLASSIFICATION: BENIGN" in response_upper or "CLASSIFICATION:BENIGN" in response_upper:
        return 0

    # Look for keywords anywhere
    has_phishing = "PHISHING" in response_upper or "SPAM" in response_upper or "MALICIOUS" in response_upper
    has_benign = "BENIGN" in response_upper or "LEGITIMATE" in response_upper or "NOT PHISHING" in response_upper or "HAM" in response_upper

    if has_phishing and not has_benign:
        return 1
    if has_benign and not has_phishing:
        return 0

    # If both present, check which comes first in context
    if has_phishing and has_benign:
        # "NOT phishing" = benign
        if "NOT PHISHING" in response_upper or "NOT A PHISHING" in response_upper:
            return 0
        # Otherwise first keyword wins
        for word in response_upper.split():
            if word in ["PHISHING", "SPAM", "MALICIOUS"]:
                return 1
            if word in ["BENIGN", "LEGITIMATE", "HAM"]:
                return 0

    return -1

def evaluate_model(model, tokenizer, label, test_data, num_samples=50):
    """Evaluate model on test set and return detailed metrics."""
    print(f"\nEvaluating: {label} (on {num_samples} samples)...")

    y_true = []
    y_pred = []
    failures = 0

    for i, (_, row) in enumerate(test_data.head(num_samples).iterrows()):
        if i % 10 == 0:
            print(f"  Processing {i+1}/{num_samples}...")

        prompt = email_to_prompt(row)
        response = ask(model, tokenizer, prompt)
        pred = extract_prediction(response)

        y_true.append(row['label'])
        if pred == -1:
            failures += 1
            y_pred.append(0)  # Default to benign if can't parse
        else:
            y_pred.append(pred)

    # Calculate metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"  Accuracy:  {acc:.3f}")
    print(f"  Precision: {prec:.3f}")
    print(f"  Recall:    {rec:.3f}")
    print(f"  F1 Score:  {f1:.3f}")
    print(f"  Parse failures: {failures}/{num_samples}")

    # Store results
    ALL_RESULTS[label] = {
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'y_true': y_true,
        'y_pred': y_pred,
        'failures': failures,
    }
    return acc, f1


def prepare_sft_dataset(tokenizer, train_data):
    """Format training data for SFT."""
    formatted = []
    for _, row in train_data.iterrows():
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT + "\n/no_think"},
            {"role": "user", "content": email_to_prompt(row)},
            {"role": "assistant", "content": email_to_response(row)},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        formatted.append({"text": text})
    return Dataset.from_list(formatted)


def setup_lora(model):
    """Configure and apply LoRA."""
    model = prepare_model_for_kbit_training(model)
    config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, config)
    trainable, total = model.get_nb_trainable_parameters()
    print(f"  LoRA trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    return model


def fine_tune(model, tokenizer, dataset, output_dir, epochs=3):
    """Run fine-tuning."""
    args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=2,   # Can do 2 with A100
        gradient_accumulation_steps=4,
        learning_rate=5e-5,              # Conservative to prevent unlearning
        weight_decay=0.01,
        warmup_steps=10,
        logging_steps=5,
        save_strategy="no",
        bf16=True,
        dataset_text_field="text",
        max_length=2048,
    )
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
    print(f"  🏋️ Training on {len(dataset)} examples for {epochs} epochs...")
    start = time.time()
    trainer.train()
    duration = time.time() - start
    print(f"  Training completed in {duration/60:.1f} minutes")
    return duration


def cleanup():
    """Free GPU memory."""
    gc.collect()
    torch.cuda.empty_cache()
    print("🧹 GPU memory cleared.")


# How many test samples to evaluate on (more = slower but more reliable)
NUM_TEST_SAMPLES = 50

print("All helper functions loaded!")
print(f"Will evaluate on {NUM_TEST_SAMPLES} test samples per model")

All helper functions loaded!
Will evaluate on 50 test samples per model


In [ ]:
# TODO: Add your Hugging Face login here before running this notebook.
# from huggingface_hub import login
# login(token="YOUR_HUGGINGFACE_TOKEN_HERE")

In [7]:
MODEL_1 = "Qwen/Qwen3-4B"

model1, tok1 = load_model(MODEL_1)

# Baseline
acc1_base, f1_1_base = evaluate_model(model1, tok1, "Qwen3-4B (baseline)", test_df, NUM_TEST_SAMPLES)

# Fine-tune
dataset1 = prepare_sft_dataset(tok1, train_df)
model1 = setup_lora(model1)
time1 = fine_tune(model1, tok1, dataset1, "./qwen3-4b-phishing")

# After fine-tuning
acc1_ft, f1_1_ft = evaluate_model(model1, tok1, "Qwen3-4B (fine-tuned)", test_df, NUM_TEST_SAMPLES)

# Show a sample response
print("\nSample fine-tuned response:")
print("-" * 40)
sample_test = test_df[test_df['label']==1].iloc[0]
print(ask(model1, tok1, email_to_prompt(sample_test)))
print("-" * 40)

del model1, tok1, dataset1
cleanup()


Loading: Qwen/Qwen3-4B


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


  🏋️ Training on 400 examples for 3 epochs...


Step,Training Loss
5,3.856507
10,3.488065
15,3.053933
20,2.651424
25,2.281790
30,2.272389
35,2.173369
40,2.010944
45,1.945165
50,2.015502


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen3DecoderLayer. Setting `past_key_values=None`.


  Training completed in 8.6 minutes

Evaluating: Qwen3-4B (fine-tuned) (on 50 samples)...
  Processing 1/50...
  Processing 11/50...
  Processing 21/50...
  Processing 31/50...
  Processing 41/50...
  Accuracy:  0.760
  Precision: 0.000
  Recall:    0.000
  F1 Score:  0.000
  Parse failures: 50/50

Sample fine-tuned response:
----------------------------------------

----------------------------------------
🧹 GPU memory cleared.


In [ ]:
MODEL_2 = "Qwen/Qwen3-8B"

model2, tok2 = load_model(MODEL_2)

# Baseline
acc2_base, f1_2_base = evaluate_model(model2, tok2, "Qwen3-8B (baseline)", test_df, NUM_TEST_SAMPLES)

# Fine-tune
dataset2 = prepare_sft_dataset(tok2, train_df)
model2 = setup_lora(model2)
time2 = fine_tune(model2, tok2, dataset2, "./qwen3-8b-phishing")

# After fine-tuning
acc2_ft, f1_2_ft = evaluate_model(model2, tok2, "Qwen3-8B (fine-tuned)", test_df, NUM_TEST_SAMPLES)

print("\n📝 Sample fine-tuned response:")
print("-" * 40)
sample_test = test_df[test_df['label']==1].iloc[0]
print(ask(model2, tok2, email_to_prompt(sample_test)))
print("-" * 40)

del model2, tok2, dataset2
cleanup()


Loading: Qwen/Qwen3-8B


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Truncating train dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


  🏋️ Training on 400 examples for 3 epochs...


Step,Training Loss
5,3.678836
10,3.351742
15,2.895318
20,2.562511
25,2.210806
30,2.184979
35,2.109504
40,1.910061
45,1.855431
50,1.917042


  Training completed in 8.9 minutes

Evaluating: Qwen3-8B (fine-tuned) (on 50 samples)...
  Processing 1/50...
  Processing 11/50...
  Processing 21/50...


In [1]:
MODEL_3 = "mistralai/Mistral-7B-Instruct-v0.3"

model3, tok3 = load_model(MODEL_3)

# Baseline
acc3_base, f1_3_base = evaluate_model(model3, tok3, "Mistral-7B (baseline)", test_df, NUM_TEST_SAMPLES)

# Fine-tune
dataset3 = prepare_sft_dataset(tok3, train_df)
model3 = setup_lora(model3)
time3 = fine_tune(model3, tok3, dataset3, "./mistral-7b-phishing")

# After fine-tuning
acc3_ft, f1_3_ft = evaluate_model(model3, tok3, "Mistral-7B (fine-tuned)", test_df, NUM_TEST_SAMPLES)

print("\n📝 Sample fine-tuned response:")
print("-" * 40)
sample_test = test_df[test_df['label']==1].iloc[0]
print(ask(model3, tok3, email_to_prompt(sample_test)))
print("-" * 40)

del model3, tok3, dataset3
cleanup()

NameError: name 'load_model' is not defined

In [ ]:
MODEL_4 = "meta-llama/Llama-3.1-8B-Instruct"

model4, tok4 = load_model(MODEL_4)

# Baseline
acc4_base, f1_4_base = evaluate_model(model4, tok4, "Llama-3.1-8B (baseline)", test_df, NUM_TEST_SAMPLES)

# Fine-tune
dataset4 = prepare_sft_dataset(tok4, train_df)
model4 = setup_lora(model4)
time4 = fine_tune(model4, tok4, dataset4, "./llama-8b-phishing")

# After fine-tuning
acc4_ft, f1_4_ft = evaluate_model(model4, tok4, "Llama-3.1-8B (fine-tuned)", test_df, NUM_TEST_SAMPLES)

print("\n📝 Sample fine-tuned response:")
print("-" * 40)
sample_test = test_df[test_df['label']==1].iloc[0]
print(ask(model4, tok4, email_to_prompt(sample_test)))
print("-" * 40)

del model4, tok4, dataset4
cleanup()

In [ ]:
print("=" * 80)
print("📊 MULTI-MODEL COMPARISON — PHISHING EMAIL DETECTION FINE-TUNING")
print("=" * 80)
print(f"Dataset: SpamAssassin ({len(df)} total emails)")
print(f"Training: {len(train_df)} examples | Testing: {NUM_TEST_SAMPLES} examples")
print(f"Method: QLoRA (4-bit, LoRA r=16, lr=5e-5, 3 epochs)")
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print()

# Build comparison table
models = [
    ("Qwen3-4B", "Qwen3-4B (baseline)", "Qwen3-4B (fine-tuned)"),
    ("Qwen3-8B", "Qwen3-8B (baseline)", "Qwen3-8B (fine-tuned)"),
    ("Mistral-7B", "Mistral-7B (baseline)", "Mistral-7B (fine-tuned)"),
    ("Llama-3.1-8B", "Llama-3.1-8B (baseline)", "Llama-3.1-8B (fine-tuned)"),
]

# Header
print(f"{'Model':<18} │ {'Baseline':^25} │ {'Fine-tuned':^25} │ {'Δ F1':^8}")
print(f"{'':18} │ {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} │ {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} │")
print("─" * 80)

for name, base_key, ft_key in models:
    if base_key in ALL_RESULTS and ft_key in ALL_RESULTS:
        b = ALL_RESULTS[base_key]
        f = ALL_RESULTS[ft_key]
        delta_f1 = f['f1'] - b['f1']
        delta_str = f"+{delta_f1:.3f}" if delta_f1 >= 0 else f"{delta_f1:.3f}"
        print(f"{name:<18} │ {b['accuracy']:.3f}  {b['precision']:.3f}  {b['recall']:.3f}  {b['f1']:.3f}  │ {f['accuracy']:.3f}  {f['precision']:.3f}  {f['recall']:.3f}  {f['f1']:.3f}  │ {delta_str}")
    else:
        print(f"{name:<18} │ {'(not run)':^25} │ {'(not run)':^25} │")

print("─" * 80)

# Per-model detailed reports
print("\n\n📋 DETAILED CLASSIFICATION REPORTS")
print("=" * 80)

for name, base_key, ft_key in models:
    for key in [base_key, ft_key]:
        if key in ALL_RESULTS:
            r = ALL_RESULTS[key]
            print(f"\n--- {key} ---")
            print(classification_report(
                r['y_true'], r['y_pred'],
                target_names=['Benign', 'Phishing'],
                digits=3,
                zero_division=0
            ))
            cm = confusion_matrix(r['y_true'], r['y_pred'])
            print(f"Confusion Matrix:")
            print(f"  Predicted:    Benign  Phishing")
            print(f"  Actual Benign:  {cm[0][0]:>4}    {cm[0][1]:>4}")
            print(f"  Actual Phish:   {cm[1][0]:>4}    {cm[1][1]:>4}")


# Summary insights
print("\n\n🏆 KEY FINDINGS")
print("=" * 80)

# Find best baseline
best_base = max(
    [(name, ALL_RESULTS[bk]['f1']) for name, bk, fk in models if bk in ALL_RESULTS],
    key=lambda x: x[1]
)
print(f"Best baseline model: {best_base[0]} (F1: {best_base[1]:.3f})")

# Find best fine-tuned
best_ft = max(
    [(name, ALL_RESULTS[fk]['f1']) for name, bk, fk in models if fk in ALL_RESULTS],
    key=lambda x: x[1]
)
print(f"Best fine-tuned model: {best_ft[0]} (F1: {best_ft[1]:.3f})")

# Check for unlearning
print(f"\nUnlearning check (did fine-tuning make things worse?):")
for name, bk, fk in models:
    if bk in ALL_RESULTS and fk in ALL_RESULTS:
        delta = ALL_RESULTS[fk]['f1'] - ALL_RESULTS[bk]['f1']
        status = "✅ IMPROVED" if delta > 0 else "⚠️ UNLEARNING" if delta < 0 else "➡️ NO CHANGE"
        print(f"  {name}: {status} (F1 change: {delta:+.3f})")

print(f"\nRecommendation: Use {best_ft[0]} for the phishing detection project")
print("Next steps: Scale to 1000+ training examples, test with real website data")

In [ ]:
try:
    import matplotlib.pyplot as plt

    model_names = []
    baseline_f1s = []
    finetuned_f1s = []

    for name, bk, fk in models:
        if bk in ALL_RESULTS and fk in ALL_RESULTS:
            model_names.append(name)
            baseline_f1s.append(ALL_RESULTS[bk]['f1'])
            finetuned_f1s.append(ALL_RESULTS[fk]['f1'])

    x = np.arange(len(model_names))
    width = 0.35

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # F1 Score comparison
    ax1 = axes[0]
    bars1 = ax1.bar(x - width/2, baseline_f1s, width, label='Baseline', color='#3498db', alpha=0.8)
    bars2 = ax1.bar(x + width/2, finetuned_f1s, width, label='Fine-tuned', color='#e74c3c', alpha=0.8)
    ax1.set_ylabel('F1 Score')
    ax1.set_title('F1 Score: Baseline vs Fine-tuned')
    ax1.set_xticks(x)
    ax1.set_xticklabels(model_names, rotation=15)
    ax1.legend()
    ax1.set_ylim(0, 1.05)
    for bar in bars1:
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

    # All metrics for fine-tuned models
    ax2 = axes[1]
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    x2 = np.arange(len(metrics))
    width2 = 0.2
    for i, (name, bk, fk) in enumerate(models):
        if fk in ALL_RESULTS:
            vals = [ALL_RESULTS[fk][m] for m in metrics]
            ax2.bar(x2 + i*width2, vals, width2, label=name, alpha=0.8)
    ax2.set_ylabel('Score')
    ax2.set_title('Fine-tuned Models: All Metrics')
    ax2.set_xticks(x2 + width2 * 1.5)
    ax2.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1'])
    ax2.legend(fontsize=8)
    ax2.set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("📊 Chart saved as model_comparison.png")

except ImportError:
    print("matplotlib not available, skipping charts")